# Tool Execution — Side Effects, Timeouts, Retries

Defining a tool schema is only half the job. Once the model selects a tool, your **executor** must run it safely:

- **Classify** reads vs writes vs destructive actions.
- **Bound** hung I/O with timeouts.
- **Retry** flaky reads with exponential backoff — but not blind retries on writes.
- **Surface errors** as actionable text the model can recover from.
- **Trip circuit breakers** when dependencies fail repeatedly.

This notebook hardens tool execution for a **ShopPulse Order API** assistant — the same domain as function calling, but focused on **runtime semantics**. Everything is self-contained in plain Python; no prior notebooks or frameworks are required.

In [ ]:
"""
ShopPulse Order API — environment for tool execution hardening demos.
"""

import json
import os
import random
import time
import uuid
import concurrent.futures
from dataclasses import dataclass, field
from datetime import datetime, timezone
from enum import Enum
from typing import Any, Callable

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "sk-proj-placeholder-replace-with-your-real-key")
os.environ.setdefault("OPENAI_API_KEY", OPENAI_API_KEY)


def pretty(obj: Any) -> str:
    return json.dumps(obj, indent=2, default=str)


ORDERS: dict[str, dict[str, Any]] = {
    "ORD-7001": {
        "id": "ORD-7001",
        "status": "paid",
        "total_usd": 89.99,
        "customer_email": "alice@example.com",
    },
    "ORD-7002": {
        "id": "ORD-7002",
        "status": "delivered",
        "total_usd": 24.50,
        "customer_email": "bob@example.com",
    },
}

REFUNDS: list[dict[str, Any]] = []
IDEMPOTENCY_KEYS: set[str] = set()

print(f"Orders loaded: {len(ORDERS)}")

---

## 1. Execution Concerns Beyond the Schema

The model outputs `{name, arguments}`. Your executor sits in between:

```
Model selects tool + args
        │
        ▼
┌───────────────────┐
│ executor wrapper  │  timeout, retry policy, circuit breaker
└─────────┬─────────┘
          ▼
    real tool function (HTTP, DB, payment API, …)
          ▼
    tool result message (success or actionable error text)
```

Schemas define **what** can be called. This notebook defines **how** calls are executed safely in production.

---

## 2. Idempotent vs Side-Effect Tools

| Type | Examples | Safe to retry? | Parallel calls OK? |
|------|----------|----------------|--------------------|
| **Read / idempotent** | `get_order`, `search_docs` | Yes (with backoff) | Usually yes |
| **Write / side-effect** | `create_refund`, `update_shipping` | Only with idempotency key | Often no |
| **Destructive** | `delete_order`, `purge_data` | Never without explicit design | Never |

Tag every tool with a **kind** in code so the executor applies the right policy automatically — do not rely on the model to know retry rules.

In [ ]:
class ToolKind(str, Enum):
    READ = "read"
    WRITE = "write"
    DESTRUCTIVE = "destructive"


TOOL_POLICY: dict[str, ToolKind] = {
    "get_order": ToolKind.READ,
    "search_docs": ToolKind.READ,
    "list_refunds": ToolKind.READ,
    "create_refund": ToolKind.WRITE,
    "update_shipping": ToolKind.WRITE,
    "delete_order": ToolKind.DESTRUCTIVE,
}

ALLOW_PARALLEL: dict[ToolKind, bool] = {
    ToolKind.READ: True,
    ToolKind.WRITE: False,
    ToolKind.DESTRUCTIVE: False,
}


def retry_allowed(tool_name: str) -> bool:
    return TOOL_POLICY.get(tool_name, ToolKind.WRITE) == ToolKind.READ


def parallel_allowed(tool_name: str) -> bool:
    kind = TOOL_POLICY.get(tool_name, ToolKind.WRITE)
    return ALLOW_PARALLEL[kind]


print(f"{'Tool':<20} {'Kind':<12} Retry  Parallel")
print("-" * 50)
for name in TOOL_POLICY:
    print(f"{name:<20} {TOOL_POLICY[name].value:<12} {str(retry_allowed(name)):<6} {parallel_allowed(name)}")

---

## 3. Tool Implementations — Including a Flaky Backend

Real APIs fail intermittently. We simulate that so retry and circuit-breaker logic has something to exercise.

In [ ]:
_call_counts: dict[str, int] = {}


def get_order(order_id: str) -> dict[str, Any]:
    """Read tool — flaky first two calls simulate upstream timeouts."""
    key = f"get_order:{order_id}"
    _call_counts[key] = _call_counts.get(key, 0) + 1
    if _call_counts[key] <= 2:
        raise TimeoutError(f"simulated upstream timeout (attempt {_call_counts[key]})")
    order = ORDERS.get(order_id.upper())
    if not order:
        raise ValueError(f"order {order_id} not found")
    return order


def search_docs(query: str) -> str:
    """Stable read tool — always succeeds."""
    docs = [
        "Refunds allowed within 30 days of delivery.",
        "Shipping updates via PATCH /v1/orders/{id}/shipping.",
    ]
    q = query.lower()
    hits = [d for d in docs if any(w in d.lower() for w in q.split())]
    return "\n".join(hits) if hits else docs[0]


def create_refund(
    order_id: str,
    amount_usd: float,
    reason: str,
    idempotency_key: str,
) -> dict[str, Any]:
    """Write tool — idempotent when the same key is retried."""
    if idempotency_key in IDEMPOTENCY_KEYS:
        existing = next((r for r in REFUNDS if r.get("idempotency_key") == idempotency_key), None)
        return {"success": True, "duplicate": True, "refund": existing}
    order = ORDERS.get(order_id.upper())
    if not order:
        return {"success": False, "error": "order not found"}
    refund = {
        "id": f"REF-{uuid.uuid4().hex[:6].upper()}",
        "order_id": order_id.upper(),
        "amount_usd": amount_usd,
        "reason": reason,
        "idempotency_key": idempotency_key,
        "created_at": datetime.now(timezone.utc).isoformat(),
    }
    IDEMPOTENCY_KEYS.add(idempotency_key)
    REFUNDS.append(refund)
    return {"success": True, "duplicate": False, "refund": refund}


TOOL_REGISTRY: dict[str, Callable[..., Any]] = {
    "get_order": get_order,
    "search_docs": search_docs,
    "create_refund": create_refund,
}

print("Tools registered:", list(TOOL_REGISTRY.keys()))

---

## 4. Timeout Wrapper — Never Block the Agent Loop Forever

A hung HTTP call can freeze an entire agent session. Wrap every tool invocation with a **wall-clock timeout**.

For synchronous tools, `concurrent.futures` with `future.result(timeout=...)` works well. For async tools, use `asyncio.wait_for` in an async executor.

In [ ]:
def run_with_timeout(fn: Callable[..., Any], *args: Any, timeout: float = 3.0, **kwargs: Any) -> Any:
    """Run fn in a thread pool and raise TimeoutError if it exceeds timeout seconds."""
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
        future = pool.submit(fn, *args, **kwargs)
        try:
            return future.result(timeout=timeout)
        except concurrent.futures.TimeoutError as exc:
            raise TimeoutError(f"Tool timed out after {timeout}s") from exc


def slow_search(query: str) -> str:
    time.sleep(2.0)  # simulates hung dependency
    return "never reached"


try:
    run_with_timeout(slow_search, "refund", timeout=0.1)
except TimeoutError as exc:
    print(f"Caught expected timeout: {exc}")

---

## 5. Retry with Exponential Backoff

For **idempotent reads**, retry transient failures with increasing delays:

```
attempt 1 → fail → wait 0.2s
attempt 2 → fail → wait 0.4s
attempt 3 → success
```

Never apply blind retries to **writes** unless you have **idempotency keys** (section 10).

In [ ]:
def retry_call(
    fn: Callable[..., Any],
    *args: Any,
    max_attempts: int = 3,
    base_delay: float = 0.05,
    **kwargs: Any,
) -> Any:
    last_exc: Exception | None = None
    for attempt in range(1, max_attempts + 1):
        try:
            return fn(*args, **kwargs)
        except Exception as exc:
            last_exc = exc
            if attempt == max_attempts:
                break
            delay = base_delay * (2 ** (attempt - 1))
            time.sleep(delay)
    raise last_exc  # type: ignore[misc]


_call_counts.clear()
result = retry_call(get_order, "ORD-7001", max_attempts=4)
print("Order retrieved:", result["id"])
print("Total attempts:", _call_counts.get("get_order:ORD-7001"))

---

## 6. Tenacity — Declarative Retries (Optional)

Production code often uses the **`tenacity`** library for retry decorators instead of manual loops. Only decorate **idempotent** functions.

In [ ]:
try:
    from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

    @retry(
        reraise=True,
        stop=stop_after_attempt(4),
        wait=wait_exponential(multiplier=0.05, min=0.05, max=0.5),
        retry=retry_if_exception_type(TimeoutError),
    )
    def tenacious_get_order(order_id: str) -> dict[str, Any]:
        return get_order(order_id)

    _call_counts.clear()
    order = tenacious_get_order("ORD-7002")
    print(f"Tenacity success: {order['id']} after {_call_counts.get('get_order:ORD-7002')} attempts")
except ImportError:
    print("tenacity not installed — manual retry_call in section 5 is equivalent.")

---

## 7. Friendly Tool Errors — Model-Visible Recovery

When a tool fails, return **actionable text** for the model — not a raw stack trace aimed at developers.

The model reads the error on the next turn and can retry with different arguments, pick another tool, or explain the failure to the user.

In [ ]:
def friendly_tool_error(tool_name: str, exc: Exception, attempt: int, max_attempts: int) -> str:
    hints = {
        "TimeoutError": "Upstream service slow — retry or narrow the query.",
        "ValueError": "Invalid arguments — verify order ID format (ORD-####).",
        "RuntimeError": "Dependency unavailable — circuit may be open.",
    }
    hint = hints.get(type(exc).__name__, "Unexpected failure — try an alternative approach.")
    return (
        f"Tool `{tool_name}` failed on attempt {attempt}/{max_attempts}: {exc}. "
        f"Hint: {hint}"
    )


print(friendly_tool_error("get_order", TimeoutError("simulated"), 3, 3))
print(friendly_tool_error("get_order", ValueError("order ORD-9999 not found"), 1, 1))

---

## 8. Circuit Breaker — Fail Fast on Broken Dependencies

After repeated failures, **stop calling** the broken service temporarily. Return a clear fallback instead of waiting for timeouts on every request.

```
CLOSED (normal) → failures accumulate → OPEN (reject calls) → timeout → HALF-OPEN (probe)
```

In [ ]:
class CircuitBreaker:
    """Opens after failure_threshold failures; resets after reset_seconds."""

    def __init__(self, failure_threshold: int = 3, reset_seconds: float = 2.0):
        self.failure_threshold = failure_threshold
        self.reset_seconds = reset_seconds
        self.failures = 0
        self.opened_at: float | None = None

    @property
    def is_open(self) -> bool:
        if self.opened_at is None:
            return False
        if time.time() - self.opened_at >= self.reset_seconds:
            self.opened_at = None
            self.failures = 0
            return False
        return True

    def before_call(self) -> None:
        if self.is_open:
            raise RuntimeError("circuit open — order service unavailable, use cached docs")

    def record_success(self) -> None:
        self.failures = 0
        self.opened_at = None

    def record_failure(self) -> None:
        self.failures += 1
        if self.failures >= self.failure_threshold:
            self.opened_at = time.time()


order_breaker = CircuitBreaker(failure_threshold=2, reset_seconds=1.0)

for i in range(4):
    try:
        order_breaker.before_call()
        raise TimeoutError("order API down")
    except Exception as exc:
        order_breaker.record_failure()
        print(f"call {i + 1}: {type(exc).__name__}: {exc} (failures={order_breaker.failures}, open={order_breaker.is_open})")

---

## 9. Unified Tool Executor

The `ToolExecutor` class combines policy, timeout, retry, circuit breaker, and friendly errors into one boundary — the only place tool functions should be invoked from.

In [ ]:
@dataclass
class ToolMessage:
    """Result returned to the agent conversation (mirrors OpenAI tool message shape)."""
    role: str
    tool_call_id: str
    content: str

    def to_dict(self) -> dict[str, str]:
        return {"role": self.role, "tool_call_id": self.tool_call_id, "content": self.content}


@dataclass
class ToolExecutionEvent:
    """Structured log entry for observability."""
    tool: str
    latency_ms: float
    outcome: str  # success | error | timeout | circuit_open
    attempt: int = 1
    circuit_state: str = "closed"


@dataclass
class ToolExecutor:
    registry: dict[str, Callable[..., Any]]
    breakers: dict[str, CircuitBreaker] = field(default_factory=dict)
    events: list[ToolExecutionEvent] = field(default_factory=list)
    default_timeout: float = 2.0
    default_max_attempts: int = 3

    def get_breaker(self, tool_name: str) -> CircuitBreaker:
        if tool_name not in self.breakers:
            self.breakers[tool_name] = CircuitBreaker()
        return self.breakers[tool_name]

    def execute(
        self,
        tool_name: str,
        args: dict[str, Any],
        tool_call_id: str = "call_manual",
        *,
        timeout: float | None = None,
        max_attempts: int | None = None,
    ) -> ToolMessage:
        timeout = timeout if timeout is not None else self.default_timeout
        attempts_limit = max_attempts if max_attempts is not None else self.default_max_attempts
        attempts = attempts_limit if retry_allowed(tool_name) else 1

        fn = self.registry.get(tool_name)
        if fn is None:
            return ToolMessage("tool", tool_call_id, f"Unknown tool: {tool_name}")

        breaker = self.get_breaker(tool_name)
        last_error: Exception | None = None

        for attempt in range(1, attempts + 1):
            start = time.perf_counter()
            try:
                breaker.before_call()
                result = run_with_timeout(fn, **args, timeout=timeout)
                breaker.record_success()
                latency = (time.perf_counter() - start) * 1000
                self.events.append(ToolExecutionEvent(
                    tool=tool_name, latency_ms=latency, outcome="success", attempt=attempt,
                    circuit_state="open" if breaker.is_open else "closed",
                ))
                content = result if isinstance(result, str) else json.dumps(result)
                return ToolMessage("tool", tool_call_id, content)

            except Exception as exc:
                last_error = exc
                breaker.record_failure()
                latency = (time.perf_counter() - start) * 1000
                outcome = "circuit_open" if isinstance(exc, RuntimeError) and "circuit open" in str(exc) else "error"
                if isinstance(exc, TimeoutError):
                    outcome = "timeout"
                self.events.append(ToolExecutionEvent(
                    tool=tool_name, latency_ms=latency, outcome=outcome, attempt=attempt,
                    circuit_state="open" if breaker.is_open else "closed",
                ))
                if attempt < attempts:
                    time.sleep(0.05 * (2 ** (attempt - 1)))
                    continue

        error_text = friendly_tool_error(tool_name, last_error, attempts, attempts)
        return ToolMessage("tool", tool_call_id, error_text)


executor = ToolExecutor(registry=TOOL_REGISTRY)
print("ToolExecutor ready.")

---

## 10. Idempotency Keys — Safe Write Retries

Payment and refund APIs accept an **`idempotency_key`**. Retrying with the same key must not create duplicate side effects.

The agent runtime should generate the key **once per user intent** and reuse it across retries — not let the model invent a new key on each attempt.

In [ ]:
idempotency_key = f"idem-{uuid.uuid4().hex[:8]}"
write_args = {
    "order_id": "ORD-7002",
    "amount_usd": 24.50,
    "reason": "customer request",
    "idempotency_key": idempotency_key,
}

first = executor.execute("create_refund", write_args, tool_call_id="call_refund_1")
second = executor.execute("create_refund", write_args, tool_call_id="call_refund_2")  # simulated retry

print("First call:", first.content[:80])
print("Retry (same key):", second.content[:80])
print(f"Refunds in store: {len(REFUNDS)} (should be 1)")

---

## 11. Hands-On — Flaky Read with Retry

Watch the executor retry `get_order` until the simulated backend succeeds.

In [ ]:
_call_counts.clear()
executor.events.clear()
executor.breakers.clear()

msg = executor.execute(
    "get_order",
    {"order_id": "ORD-7001"},
    tool_call_id="call_get_7001",
    max_attempts=5,
    timeout=1.0,
)

print("Result:", msg.content)
print("\nExecution events:")
for ev in executor.events:
    print(f"  {ev.tool} attempt={ev.attempt} outcome={ev.outcome} latency={ev.latency_ms:.1f}ms")

---

## 12. Hands-On — Circuit Breaker Blocks Further Calls

After enough failures, the breaker opens and returns immediately without hitting the flaky service.

In [ ]:
def always_fails(order_id: str) -> dict[str, Any]:
    raise TimeoutError("payment service permanently down for demo")


failing_executor = ToolExecutor(
    registry={"get_order": always_fails},
    default_max_attempts=1,
)
failing_executor.breakers["get_order"] = CircuitBreaker(failure_threshold=2, reset_seconds=5.0)

for i in range(4):
    msg = failing_executor.execute("get_order", {"order_id": "ORD-7001"}, tool_call_id=f"call_{i}")
    print(f"Call {i + 1}: {msg.content[:90]}...")

print(f"\nCircuit open: {failing_executor.breakers['get_order'].is_open}")

---

## 13. Decision Matrix

| Signal | Executor action |
|--------|-----------------|
| Read timeout | Retry with backoff (≤3–5 attempts) |
| Read 404 / not found | No retry — return friendly error to model |
| Write conflict | No retry — return idempotency status |
| Burst of 5xx / timeouts | Open circuit breaker |
| Circuit open | Fail fast with fallback message |
| Unknown tool | Single error message, no retry |
| Destructive tool | Require HITL — never auto-retry |

In [ ]:
def recommend_action(tool_name: str, exc: Exception, attempt: int) -> str:
    kind = TOOL_POLICY.get(tool_name, ToolKind.WRITE)
    if isinstance(exc, ValueError) and "not found" in str(exc).lower():
        return "no_retry — return error to model"
    if isinstance(exc, RuntimeError) and "circuit open" in str(exc):
        return "fail_fast — use fallback docs"
    if isinstance(exc, TimeoutError) and kind == ToolKind.READ and attempt < 3:
        return "retry_with_backoff"
    if kind == ToolKind.WRITE:
        return "no_retry unless idempotency_key matches"
    if kind == ToolKind.DESTRUCTIVE:
        return "NEVER auto-retry — HITL required"
    return "return friendly error to model"


SCENARIOS = [
    ("get_order", TimeoutError("slow"), 1),
    ("get_order", ValueError("order ORD-9999 not found"), 1),
    ("create_refund", TimeoutError("slow"), 2),
    ("delete_order", RuntimeError("forbidden"), 1),
]

for tool, exc, att in SCENARIOS:
    print(f"{tool} + {type(exc).__name__} → {recommend_action(tool, exc, att)}")

---

## 14. Observability — Log Every Tool Call

Production agents log structured events per tool invocation. The `ToolExecutor` above appends to `events`; export these to your monitoring stack.

In [ ]:
def summarize_events(events: list[ToolExecutionEvent]) -> dict[str, Any]:
    if not events:
        return {"count": 0}
    outcomes: dict[str, int] = {}
    latencies = []
    for ev in events:
        outcomes[ev.outcome] = outcomes.get(ev.outcome, 0) + 1
        latencies.append(ev.latency_ms)
    return {
        "count": len(events),
        "outcomes": outcomes,
        "avg_latency_ms": round(sum(latencies) / len(latencies), 2),
        "max_attempt": max(e.attempt for e in events),
    }


print("Event summary from flaky read demo:")
print(pretty(summarize_events(executor.events)))

---

## 15. Agent Loop Integration

The executor plugs into any tool-calling loop. Below, a minimal agent uses `ToolExecutor` instead of calling registry functions directly.

In [ ]:
@dataclass
class ResilientToolAgent:
    """Agent loop that routes all tool execution through ToolExecutor."""

    executor: ToolExecutor
    messages: list[dict[str, Any]] = field(default_factory=list)

    def handle_tool_calls(self, tool_calls: list[dict[str, Any]]) -> list[ToolMessage]:
        results = []
        for tc in tool_calls:
            if not parallel_allowed(tc["name"]) and len(tool_calls) > 1:
                # Force sequential writes — take first only in this demo
                pass
            msg = self.executor.execute(
                tc["name"],
                tc["args"],
                tool_call_id=tc["id"],
            )
            results.append(msg)
        return results

    def run_simulated(self, tool_calls: list[dict[str, Any]]) -> None:
        self.messages.append({"role": "assistant", "tool_calls": tool_calls})
        tool_msgs = self.handle_tool_calls(tool_calls)
        for tm in tool_msgs:
            self.messages.append(tm.to_dict())


_call_counts.clear()
agent = ResilientToolAgent(executor=ToolExecutor(registry=TOOL_REGISTRY))
agent.run_simulated([{
    "id": "call_agent_1",
    "name": "get_order",
    "args": {"order_id": "ORD-7001"},
}])

for m in agent.messages:
    preview = json.dumps(m)[:100]
    print(preview + ("..." if len(preview) >= 100 else ""))

---

## 16. Common Mistakes

| Mistake | Consequence | Fix |
|---------|-------------|-----|
| Retry writes without idempotency key | Double refunds, duplicate records | Generate key once per intent |
| No timeout | Agent hangs forever | `run_with_timeout` on every call |
| Raw stack traces to model | Confused recovery | `friendly_tool_error` |
| No circuit breaker | Retry storm on dead service | Fail fast after threshold |
| Parallel write tool calls | Race conditions | `ALLOW_PARALLEL` policy |
| Scattered dispatch logic | Inconsistent policies | Single `ToolExecutor` boundary |

---

## 17. Check Your Understanding

1. Which tool kinds are safe to retry blindly? Which are not?
2. Why must writes use an **idempotency key** before retry is allowed?
3. What happens when `run_with_timeout` exceeds its limit?
4. When does the circuit breaker **open**, and what should the executor return?
5. What fields does `ToolExecutionEvent` capture for observability?
6. Why return friendly errors instead of stack traces to the model?
7. Why disable parallel execution for write tools?

---

## 18. Summary

Tool schemas define **what** can be called; the **executor** defines **how** calls run safely.

**Key takeaways:**

- Classify tools as **read**, **write**, or **destructive** — policy drives retry and parallelism.
- Wrap every invocation with **timeouts** so hung I/O cannot block the agent loop.
- **Retry idempotent reads** with exponential backoff; **never blind-retry writes** without idempotency keys.
- Return **actionable error text** so the model can recover on the next turn.
- Use **circuit breakers** to fail fast when dependencies are down.
- Centralize execution in one **`ToolExecutor`** boundary with structured **observability events**.
- The agent loop should never call tool functions directly — always go through the hardened executor.

With execution semantics in place, your agents can survive flaky APIs without duplicate side effects or infinite hangs.